# Chameleon Setup (Thin IaC Runner)

This notebook now does only the steps that still need to happen outside the repo:

1. Create the Chameleon lease and VM
2. Open the required ports, attach a floating IP, and optionally reattach an existing block volume
3. Install Docker and K3s if needed
4. Clone this monorepo onto the VM
5. Print the VM-side secrets/bootstrap commands for the TA
6. SSH to the VM and run `scripts/bootstrap-argocd.sh` there

All Kubernetes manifests and deployment logic now live in the repo under `k8s/` and `scripts/`.

Before running this notebook:
- Make sure the branch in `REPO_BRANCH` contains the latest deploy changes
- The notebook stops after clone and prints the exact VM commands to run next
- If you are recreating Sharvin's VM, enable `ATTACH_EXISTING_BLOCK_VOLUME` in the config cell before running the recovery cell


In [ ]:
from chi import server, context, lease
import chi
import openstack
import os
import time
import base64
import datetime
from pathlib import Path
from textwrap import dedent

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
print(f"Logged in as: {username}")

LEASE_NAME = f"proj18-mealie2"
LEASE_DURATION_HOURS = 10
VM_NAME = f"node-iac2"
VM_IMAGE = "CC-Ubuntu24.04"
VM_FLAVOR = "m1.xlarge"

REPO_URL = "https://github.com/brycemiranda/proj18-bias-variance.git"
REPO_BRANCH = "argocd-bootstrap-path"
REMOTE_DIR = "/home/cc/proj18-bias-variance"
LOCAL_SECRETS_FILE = Path("scripts/secrets.env")

# Optional recovery profile for Sharvin's rebuilt VM.
ATTACH_EXISTING_BLOCK_VOLUME = True
EXISTING_BLOCK_VOLUME_NAME = "block-proj18-sns10089_nyu_edu"
EXISTING_BLOCK_VOLUME_ID = "282d70fc-d16b-436d-8a5a-e6b2fa40e5c4"
BLOCK_VOLUME_SITE = "KVM@TACC"
BLOCK_VOLUME_DEVICE = "/dev/vdb"
BLOCK_VOLUME_MOUNT_POINT = "/mnt/blockA"

OBJECT_STORAGE_BUCKET = "object-proj18-sns10089_nyu_edu"
OBJECT_STORAGE_ENDPOINT = "https://chi.tacc.chameleoncloud.org:7480"
OBJECT_STORAGE_SITE = "CHI@TACC"
OBJECT_STORAGE_ARTIFACT_KEYS = [
    "artifacts/tag_to_vector.pkl",
    "artifacts/mappings.json",
]

# Internal dev credentials for the ephemeral grading VM. Override them here or
# # provide LOCAL_SECRETS_FILE if you want different values.
DB_USERNAME = "mealie"
DB_PASSWORD = "mealie_pass"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
GRAFANA_ADMIN_PASSWORD = "admin123"


# # Set these to False for faster redeploys after the initial bootstrap.
# RUN_INGESTION_JOB = True
# RUN_BATCH_BOOTSTRAP = False
# RUN_TRAIN_BOOTSTRAP = False

# OPEN_PORTS = {
#     "allow-ssh": 22,
#     "allow-30090": 30090,
#     "allow-30800": 30800,
#     "allow-30500": 30500,
#     "allow-30900": 30900,
#     "allow-30901": 30901,
#     "allow-30091": 30091,
#     "allow-30300": 30300,
#     "allow-30903": 30903,
# }


Logged in as: sg9469_nyu_edu


## Create Lease


In [3]:
l = lease.Lease(
    LEASE_NAME,
    duration=datetime.timedelta(hours=LEASE_DURATION_HOURS),
)
l.add_flavor_reservation(id=chi.server.get_flavor_id(VM_FLAVOR), amount=1)
l.submit(idempotent=True)
l.show()

print("Lease status:", l.status)
if l.status != "ACTIVE":
    print("Lease is not ACTIVE yet. Wait a bit and rerun this cell.")
else:
    print("Lease is ACTIVE.")


Waiting for lease to start...


Lease proj18-mealie2 has reached status active


HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>proj18-mealie2…

Lease Details:
Name: proj18-mealie2
ID: be5a60b4-1c59-4dcd-86f8-184336afaa8f
Status: ACTIVE
Start Date: 2026-04-27 23:11:00
End Date: 2026-04-28 09:11:00
User ID: 55828c46b9d77d1109d40a2300dcf2b735f1ec72a114fbbc86e1e24370664091
Project ID: 89f528973fea4b3a981f9b2344e522de

Node Reservations:

Floating IP Reservations:

Network Reservations:

Flavor Reservations:
ID: a236090f-844e-48c2-a314-9f6fd4aad892, Status: active, Flavor: a236090f-844e-48c2-a314-9f6fd4aad892, Amount: 1

Events:
Lease status: ACTIVE
Lease is ACTIVE.


## Launch VM


In [13]:
conn = openstack.connect(cloud="envvars")

reserved_flavor = l.get_reserved_flavors()[0].name
print(f"Using reserved flavor: {reserved_flavor}")

s = server.Server(
    VM_NAME,
    image_name=VM_IMAGE,
    flavor_name=reserved_flavor,
)

print(f"Creating VM: {VM_NAME}")
s.submit(idempotent=True)
s.refresh()
server_id = s.id
print(f"Server ID: {server_id}")

print(f"Current VM status: {s.status}")
if s.status == "ERROR":
    raise RuntimeError(f"VM creation failed: {getattr(s, 'fault', None)}")
if s.status != "ACTIVE":
    print("VM is not ACTIVE yet. Wait a bit and rerun this cell.")
else:
    print("VM is ACTIVE")

s.refresh()
s.show(type="widget")


Using reserved flavor: reservation:a236090f-844e-48c2-a314-9f6fd4aad892
Creating VM: node-iac2


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server node-iac2's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-iac2
Id,5bda6aa7-899f-462e-b620-16ff5c5da101
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:a236090f-844e-48c2-a314-9f6fd4aad892
Addresses,sharednet1: IP: 10.56.0.82 (v4) Type: fixed MAC: fa:16:3e:08:cc:0e
Network Name,sharednet1
Created At,2026-04-28T01:22:01Z
Keypair,trovi-mwalie
Reservation Id,None
Host Id,a456a0ae4822aa6387896353c45a22b14ff7b2158367d06cfaa7eb80


Server ID: 5bda6aa7-899f-462e-b620-16ff5c5da101
Current VM status: ACTIVE
VM is ACTIVE


Attribute,node-iac2
Id,5bda6aa7-899f-462e-b620-16ff5c5da101
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:a236090f-844e-48c2-a314-9f6fd4aad892
Addresses,sharednet1: IP: 10.56.0.82 (v4) Type: fixed MAC: fa:16:3e:08:cc:0e
Network Name,sharednet1
Created At,2026-04-28T01:22:01Z
Keypair,trovi-mwalie
Reservation Id,None
Host Id,a456a0ae4822aa6387896353c45a22b14ff7b2158367d06cfaa7eb80


## Optional Recovery From Existing Block And Object Storage

Use this path when recreating Sharvin's VM and you want to recover persisted platform state instead of starting from a blank VM.

Recovery inventory:
- Block volume: `block-proj18-sns10089_nyu_edu` (`282d70fc-d16b-436d-8a5a-e6b2fa40e5c4`) on `KVM@TACC`
- Mount point target: `/mnt/block`
- Object storage bucket: `object-proj18-sns10089_nyu_edu`
- Object storage endpoint: `https://chi.tacc.chameleoncloud.org:7480`
- Backed-up artifacts: `artifacts/tag_to_vector.pkl`, `artifacts/mappings.json`

The notebook already automates repo clone and the tracked deploy flow later on. The extra recovery-specific step here is reattaching the existing block volume to the new VM.


In [6]:
if ATTACH_EXISTING_BLOCK_VOLUME:
    recovery_conn = chi.clients.connection()

    print(
        f"Attaching existing block volume {EXISTING_BLOCK_VOLUME_NAME} "
        f"({EXISTING_BLOCK_VOLUME_ID}) to server {server_id}..."
    )

    current_attachments = list(recovery_conn.compute.volume_attachments(server=server_id))
    already_attached = any(
        getattr(attachment, "volume_id", None) == EXISTING_BLOCK_VOLUME_ID
        for attachment in current_attachments
    )

    if already_attached:
        print("Volume is already attached to this VM.")
    else:
        attachment = recovery_conn.compute.create_volume_attachment(
            server=server_id,
            volume_id=EXISTING_BLOCK_VOLUME_ID,
        )
        print("Volume attachment requested:", attachment)
        time.sleep(10)

    refreshed_attachments = list(recovery_conn.compute.volume_attachments(server=server_id))
    print("Current volume attachments:")
    for attachment in refreshed_attachments:
        print(
            f"- volume_id={getattr(attachment, 'volume_id', None)} "
            f"device={getattr(attachment, 'device', None)}"
        )

    print("\nAfter SSH, verify and mount the recovered volume if needed:")
    print(f"  sudo mkdir -p {BLOCK_VOLUME_MOUNT_POINT}")
    print("  lsblk")
    print(f"  sudo mount {BLOCK_VOLUME_DEVICE}1 {BLOCK_VOLUME_MOUNT_POINT}")
else:
    print("ATTACH_EXISTING_BLOCK_VOLUME=False; skipping existing block-volume attachment.")

print("\nObject storage backup details:")
print(f"- bucket:   {OBJECT_STORAGE_BUCKET}")
print(f"- endpoint: {OBJECT_STORAGE_ENDPOINT}")
print(f"- site:     {OBJECT_STORAGE_SITE}")
print("- keys:")
for key in OBJECT_STORAGE_ARTIFACT_KEYS:
    print(f"  - {key}")


Attaching existing block volume block-proj18-sns10089_nyu_edu (282d70fc-d16b-436d-8a5a-e6b2fa40e5c4) to server d63bd261-621b-41d3-8e7b-fc713d5c3a80...


BadRequestException: BadRequestException: 400: Client Error for url: https://kvm.tacc.chameleoncloud.org:8774/v2.1/servers/d63bd261-621b-41d3-8e7b-fc713d5c3a80/os-volume_attachments, Invalid volume: volume 282d70fc-d16b-436d-8a5a-e6b2fa40e5c4 is already attached to instances: c5f68f81-9841-4186-9a98-949a1a60cd77

## Open Ports And Attach Floating IP


In [14]:
from chi import network

security_groups = [
    {'name': f'{VM_NAME}-allow-ssh',   'port': 22,    'description': 'SSH access'},
    {'name': f'{VM_NAME}-allow-30090', 'port': 30090, 'description': 'Mealie UI'},
    {'name': f'{VM_NAME}-allow-30443', 'port': 30443, 'description': 'ArgoCD UI'},
    {'name': f'{VM_NAME}-allow-30800', 'port': 30800, 'description': 'Inference API'},
    {'name': f'{VM_NAME}-allow-30500', 'port': 30500, 'description': 'MLflow'},
    {'name': f'{VM_NAME}-allow-30900', 'port': 30900, 'description': 'MinIO API'},
    {'name': f'{VM_NAME}-allow-30901', 'port': 30901, 'description': 'MinIO Console'},
    {'name': f'{VM_NAME}-allow-30091', 'port': 30091, 'description': 'Prometheus'},
    {'name': f'{VM_NAME}-allow-30300', 'port': 30300, 'description': 'Grafana'},
    {'name': f'{VM_NAME}-allow-30903', 'port': 30903, 'description': 'Alertmanager'},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        'name': sg['name'],
        'description': sg['description'],
    })
    secgroup.add_rule(direction='ingress', protocol='tcp', port=sg['port'])
    secgroup.submit(idempotent=True)

    try:
        s.add_security_group(sg['name'])
        print(f"Added security group: {sg['name']}")
    except Exception as e:
        print(f"Security group attach skipped for {sg['name']}: {e}")

print("All security groups attached.")

s.associate_floating_ip()
s.refresh()
s.check_connectivity()
s.refresh()

floating_ip = None
for addr_list in s.addresses.values():
    for addr in addr_list:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]
            break

if floating_ip is None:
    raise RuntimeError("Could not detect floating IP. Check s.show() output.")

print("Floating IP:", floating_ip)
print("SSH command: ssh cc@" + floating_ip)

s.show(type="widget")


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Added security group: node-iac2-allow-ssh


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Added security group: node-iac2-allow-30090


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


...

Added security group: node-iac2-allow-30300


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Added security group: node-iac2-allow-30903
All security groups attached.
Checking connectivity to 129.114.26.157 port 22.


Connection successful
Floating IP: 129.114.26.157
SSH command: ssh cc@129.114.26.157


Attribute,node-iac2
Id,5bda6aa7-899f-462e-b620-16ff5c5da101
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:a236090f-844e-48c2-a314-9f6fd4aad892
Addresses,sharednet1: IP: 10.56.0.82 (v4) Type: fixed MAC: fa:16:3e:08:cc:0e IP: 129.114.26.157 (v4) Type: floating MAC: fa:16:3e:08:cc:0e
Network Name,sharednet1
Created At,2026-04-28T01:22:01Z
Keypair,trovi-mwalie
Reservation Id,None
Host Id,a456a0ae4822aa6387896353c45a22b14ff7b2158367d06cfaa7eb80


## Install Docker And K3s


In [17]:
docker_result = s.execute("docker --version 2>/dev/null || echo NOT_INSTALLED")
print(docker_result)
if "NOT_INSTALLED" in str(docker_result):
    print("Installing Docker...")
    s.execute("curl -sSL https://get.docker.com/ | sudo sh")
    s.execute("sudo groupadd -f docker && sudo usermod -aG docker $USER")
    s.execute("sudo systemctl restart docker")
else:
    print("Docker already installed.")

kubectl_result = s.execute("kubectl version --client 2>/dev/null || echo NOT_INSTALLED")
print(kubectl_result)
if "NOT_INSTALLED" in str(kubectl_result):
    print("Installing K3s...")
    s.execute("curl -sfL https://get.k3s.io | sudo sh -")
    time.sleep(30)
    print(s.execute("sudo systemctl status k3s --no-pager | head -5"))
else:
    print("kubectl already available.")

s.execute("sudo chmod 644 /etc/rancher/k3s/k3s.yaml")
s.execute("grep -qxF 'export KUBECONFIG=/etc/rancher/k3s/k3s.yaml' ~/.bashrc || echo 'export KUBECONFIG=/etc/rancher/k3s/k3s.yaml' >> ~/.bashrc")
print(s.execute("export KUBECONFIG=/etc/rancher/k3s/k3s.yaml && sudo kubectl get nodes"))


Docker version 29.4.1, build 055a478
Command exited with status 0.
=== stdout ===
Docker version 29.4.1, build 055a478

(no stderr)
Docker already installed.
Client Version: v1.34.6+k3s1
Kustomize Version: v5.7.1
Command exited with status 0.
=== stdout ===
Client Version: v1.34.6+k3s1
Kustomize Version: v5.7.1

(no stderr)
kubectl already available.
NAME        STATUS   ROLES           AGE   VERSION
node-iac2   Ready    control-plane   81s   v1.34.6+k3s1
Command exited with status 0.
=== stdout ===
NAME        STATUS   ROLES           AGE   VERSION
node-iac2   Ready    control-plane   81s   v1.34.6+k3s1

(no stderr)


## Clone The Monorepo

This notebook clones the GitHub branch defined in `REPO_BRANCH`. Push your local IaC changes first if they are not already on that branch.


In [18]:
clone_cmd = dedent(
    f"""
    bash -lc '
    set -e
    mkdir -p /home/cc
    if [ -d {REMOTE_DIR}/.git ]; then
      git -C {REMOTE_DIR} fetch origin
      git -C {REMOTE_DIR} checkout {REPO_BRANCH}
      git -C {REMOTE_DIR} pull --ff-only origin {REPO_BRANCH}
    else
      git clone --single-branch --branch {REPO_BRANCH} {REPO_URL} {REMOTE_DIR}
    fi
    cd {REMOTE_DIR}
    git rev-parse --abbrev-ref HEAD
    git rev-parse HEAD
    '
    """
).strip()
print(s.execute(clone_cmd))


Already on 'argocd-bootstrap-path'


Your branch is up to date with 'origin/argocd-bootstrap-path'.


From https://github.com/brycemiranda/proj18-bias-variance
 * branch            argocd-bootstrap-path -> FETCH_HEAD


Already up to date.
argocd-bootstrap-path
08d8684e7d7dbd0b57f6cd4a7b44cd49d2d45c2a
Command exited with status 0.
=== stdout ===
Your branch is up to date with 'origin/argocd-bootstrap-path'.
Already up to date.
argocd-bootstrap-path
08d8684e7d7dbd0b57f6cd4a7b44cd49d2d45c2a

=== stderr ===
Already on 'argocd-bootstrap-path'
From https://github.com/brycemiranda/proj18-bias-variance
 * branch            argocd-bootstrap-path -> FETCH_HEAD



## SSH To The VM And Run Bootstrap There

After clone, stop using the notebook for deployment.
SSH to the VM, create `scripts/secrets.env`, then run the tracked bootstrap command there.


In [ ]:
from textwrap import dedent

instructions = dedent(
    f"""
    SSH to the VM:
      ssh cc@{floating_ip}

    Then run on the VM:
      cd {REMOTE_DIR}
      git checkout {REPO_BRANCH}
      git pull --ff-only origin {REPO_BRANCH}
      git submodule update --init --recursive
      cat > scripts/secrets.env <<'EOF'
      DB_USERNAME={DB_USERNAME}
      DB_PASSWORD={DB_PASSWORD}
      MINIO_ACCESS_KEY={MINIO_ACCESS_KEY}
      MINIO_SECRET_KEY={MINIO_SECRET_KEY}
      GRAFANA_ADMIN_PASSWORD={GRAFANA_ADMIN_PASSWORD}
      EOF
      bash scripts/bootstrap-argocd.sh {floating_ip}
    """
).strip()

print(instructions)
